IMPORTS + CONFIG

In [21]:
import shutil

In [22]:
import os
# Set Kaggle credentials directly

# os.environ['KAGGLE_USERNAME'] = "danielbarcohen"
os.environ['KAGGLE_TOKEN'] = "KGAT_ea9be303bd1add4232ee54944d94dec4"

import time
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

# -----------------------------------
# KAGGLE CONFIG
# -----------------------------------

# Initialize and authenticate the API (requires kaggle.json in the standard location)
api = KaggleApi()
try:
    api.authenticate()
except Exception as e:
    print(f"KAGGLE_TOKEN ERROR: {e}")

# Kaggle search queries (removed 'extension:ipynb' as the API filters by type)
QUERIES = [
    'train_test_split',
    'sklearn.preprocessing',
    'LabelEncoder',
    'OneHotEncoder',
    'pandas',
    'pandas.read_csv',
    'RandomForestClassifier',
    'XGBClassifier',
    'feature engineering',
    'data cleaning',
    'data cleansing',
    'date prep',
    'exploration',
    'EDA'
]

MAX_NOTEBOOKS_PER_QUERY = 600

KAGGLE_NOTEBOOKS = 'kaggle_notebooks_new'

SAVE_DIR = Path(KAGGLE_NOTEBOOKS)
SAVE_DIR.mkdir(exist_ok=True)
TEMP_DIR = SAVE_DIR / "_temp"
# -----------------------------------
# CRAWL KAGGLE NOTEBOOKS
# -----------------------------------


Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
KAGGLE_TOKEN ERROR: name 'exit' is not defined


In [23]:
def crawl_kaggle_notebooks_flat():
    downloaded_notebooks = set()

    for query in QUERIES:
        time.sleep(15)
        print(f"\n==========================================")
        print(f"SEARCH QUERY: {query}")
        print(f"==========================================")

        downloaded = 0
        page = 1

        while downloaded < MAX_NOTEBOOKS_PER_QUERY:
            if downloaded % 50 == 0:
                time.sleep(15)
            try:
                kernels = api.kernels_list(
                    search=query,
                    sort_by="voteCount",
                    page=page
                )
            except Exception as e:
                print(f"Kaggle API ERROR: {e}")
                break

            if not kernels:
                print("No more results for this query.")
                break

            for kernel in kernels:
                kernel_ref = kernel.ref  # Format: 'username/kernel-name'

                if kernel_ref in downloaded_notebooks:
                    continue

                downloaded_notebooks.add(kernel_ref)

                # Clean temp directory for download
                if TEMP_DIR.exists():
                    shutil.rmtree(TEMP_DIR)
                TEMP_DIR.mkdir(exist_ok=True)

                try:
                    # 1. Pull into temporary folder
                    api.kernels_pull(
                        kernel_ref,
                        path=str(TEMP_DIR),
                        metadata=False,
                        quiet=True
                    )

                    # 2. Find the .ipynb file
                    ipynb_files = list(TEMP_DIR.glob("*.ipynb"))

                    if ipynb_files:
                        # 3. Create flat filename (e.g., "username_kernel-name.ipynb")
                        flat_name = f"{kernel_ref.replace('/', '_')}.ipynb"
                        target_file = SAVE_DIR / flat_name

                        # 4. Move directly to kaggle_notebooks/
                        shutil.move(str(ipynb_files[0]), str(target_file))

                        downloaded += 1
                        print(f"[{downloaded}/{MAX_NOTEBOOKS_PER_QUERY}] Saved flat: {flat_name}")

                except Exception as e:
                    print(f"FAILED {kernel_ref}: {e}")
                    time.sleep(3)

                # Clean up temp folder
                if TEMP_DIR.exists():
                    shutil.rmtree(TEMP_DIR)

                if downloaded >= MAX_NOTEBOOKS_PER_QUERY:
                    break

                time.sleep(1)

            page += 1

    # Final cleanup of temp directory
    if TEMP_DIR.exists():
        shutil.rmtree(TEMP_DIR)

    print("\nSUCCESS: All notebooks saved directly as flat .ipynb files!")

In [24]:
crawl_kaggle_notebooks_flat()


SEARCH QUERY: data cleaning
[1/200] Saved flat: colinmorris_exercise-strings-and-dictionaries.ipynb
[2/200] Saved flat: alexisbcook_exercise-handling-missing-values.ipynb
[3/200] Saved flat: alexisbcook_exercise-scaling-and-normalization.ipynb
[4/200] Saved flat: alexisbcook_exercise-parsing-dates.ipynb
[5/200] Saved flat: alexisbcook_exercise-character-encodings.ipynb
[6/200] Saved flat: alexisbcook_exercise-inconsistent-data-entry.ipynb
[7/200] Saved flat: pmarcelino_comprehensive-data-exploration-with-python.ipynb
[8/200] Saved flat: kanncaa1_data-sciencetutorial-for-beginners.ipynb
[9/200] Saved flat: arthurtok_introduction-to-ensembling-stacking-in-python.ipynb
[10/200] Saved flat: rtatman_data-cleaning-challenge-handling-missing-values.ipynb
[11/200] Saved flat: serigne_stacked-regressions-top-4-on-leaderboard.ipynb
[12/200] Saved flat: ldfreeman3_a-data-science-framework-to-achieve-99-accuracy.ipynb
[13/200] Saved flat: willkoehrsen_start-here-a-gentle-introduction.ipynb
[14/20

RESULTS